In [ ]:
# imports

import numpy as np
import time
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import torch.nn as nn
import pandas as pd

import random
import sklearn

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset, random_split, TensorDataset
from torchvision import datasets, transforms, models
from collections import defaultdict


# For SVM

import torch
import torchvision
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
# Dataset 1 (initial dataset)

!unzip "/content/archive.zip" # Rename to whatever you call the data

In [ ]:
path = '/content/dataset'

# Transform data

transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # Numbers from ImageNet
])

# Load dataset

dataset = torchvision.datasets.ImageFolder(root=path, transform=transform)
torch.manual_seed(42) # Ensure same seed for reproducible results
train_set, val_set = random_split(dataset, [0.889, 0.111]) # 80, 10, 10 split

print(f'Training Set Size: {len(train_set)}')
print(f'Validation Set Size: {len(val_set)}')

img, label = train_set[0]
img = img.permute(1, 2, 0)
plt.title("First Training Sample")
plt.imshow(img)
plt.show()

In [ ]:
# Dataset 2 (test dataset)

!unzip "/content/archive (2).zip"

In [ ]:
import os
import shutil
import pandas as pd
import ast
from torchvision import datasets, transforms
from torch.utils.data import Subset
from collections import defaultdict
import random

dataset_images = '/content/ODIR-5K/ODIR-5K/Training Images'
csv_path = '/content/full_df.csv'
output_path = '/content/test_set'

df = pd.read_csv(csv_path)

# Label Mapping

label_map = {
    "N": "normal",
    "D": "diabetic_retinopathy",
    "G": "glaucoma",
    "C": "cataract"
}

# Create Folders

for folder in label_map.values():
    os.makedirs(os.path.join(output_path, folder), exist_ok=True)

# Copy Images

for _, row in df.iterrows():
    labels = ast.literal_eval(row['labels'])
    filename = os.path.basename(row['filepath'])
    src_path = os.path.join(dataset_images, filename)

    for label in labels:
        if label in label_map:
            folder_name = label_map[label]
            dest_path = os.path.join(output_path, folder_name, filename)
            shutil.copy2(src_path, dest_path)

print(f"Test dataset created at: {output_path}")

# Transform data

transform = transforms.Compose([
    transforms.Resize(512),
    transforms.CenterCrop(512),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # Numbers from ImageNet
])

full_test_set = datasets.ImageFolder(root=output_path, transform=transform)

# Test set to val set matching lengths

test_set = Subset(full_test_set, range(len(val_set)))

torch.manual_seed(42) # Ensure same seed for reproducible results

print(f"Testing Set Size: {len(test_set)}")

img, label = test_set[0]
img = img.permute(1, 2, 0)
plt.title("First Testing Sample")
plt.imshow(img)
plt.show()

In [ ]:
# POTENTIAL BALANCED CLASS CODE

import os
import shutil
import pandas as pd
import ast
from torchvision import datasets, transforms
from torch.utils.data import Subset
from collections import defaultdict
import random

dataset_images = '/content/ODIR-5K/ODIR-5K/Training Images'
csv_path = '/content/full_df.csv'
output_path = '/content/test_set'

df = pd.read_csv(csv_path)

# Label Mapping

label_map = {
    "N": "normal",
    "D": "diabetic_retinopathy",
    "G": "glaucoma",
    "C": "cataract"
}

# Create Folders

for folder in label_map.values():
    os.makedirs(os.path.join(output_path, folder), exist_ok=True)

# Copy Images

for _, row in df.iterrows():
    labels = ast.literal_eval(row['labels'])
    filename = os.path.basename(row['filepath'])
    src_path = os.path.join(dataset_images, filename)

    for label in labels:
        if label in label_map:
            folder_name = label_map[label]
            dest_path = os.path.join(output_path, folder_name, filename)
            shutil.copy2(src_path, dest_path)

print(f"Test dataset created at: {output_path}")

# Transform data

transform = transforms.Compose([
    transforms.Resize(512),
    transforms.CenterCrop(512),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # Numbers from ImageNet
])

full_test_set = datasets.ImageFolder(root=output_path, transform=transform)

# Group indices by class

class_indices = defaultdict(list)
for idx, (_, label) in enumerate(full_test_set):
    class_indices[label].append(idx)

# Number to take from each class = 1/4 of the total desired size

total_size = len(full_test_set)
per_class = total_size // 4

# Sample exactly per_class from each category

balanced_indices = []
for idx_list in class_indices.values():
    balanced_indices.extend(random.sample(idx_list, per_class))

# Create balanced subset

test_set = Subset(full_test_set, balanced_indices)

torch.manual_seed(42) # Ensure same seed for reproducible results

print(f"Testing Set Size: {len(test_set)}")

img, label = test_set[0]
img = img.permute(1, 2, 0)
plt.title("First Testing Sample")
plt.imshow(img)
plt.show()

In [ ]:
# Dataset 3 (new test dataset)

!unzip "/content/new_test.zip"

In [ ]:
import os
import shutil
import random
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

number_samples = len(val_set) // 4
extract_dir = '/content/new_test'
output_dir = '/content/full_test_set'


folder_map = {
    "11.Normal Fundus": "normal",
    "4.Moderate DR": "diabetic_retinopathy",
    "10.Glaucoma": "glaucoma",
    "7.Cataract": "cataract"
}

# Make directories

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
os.makedirs(output_dir, exist_ok=True)
for new_name in folder_map.values():
    os.makedirs(os.path.join(output_dir, new_name), exist_ok=True)

# Copy images

for old_name, new_name in folder_map.items():
    old_path = os.path.join(extract_dir, old_name)
    new_path = os.path.join(output_dir, new_name)

    images = os.listdir(old_path)

    random.shuffle(images)
    for image in images[:number_samples]:
        src_path = os.path.join(old_path, image)
        dest_path = os.path.join(new_path, image)
        shutil.copy2(src_path, dest_path)

print(f"Dataset created at {output_dir}")

transform = transforms.Compose([
    transforms.Resize(512),
    transforms.CenterCrop(512),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # Numbers from ImageNet
])

new_set = datasets.ImageFolder(root=output_dir, transform=transform)

print(f"New Testing Set Size: {len(new_set)}")

img, label = new_set[0]
img = img.permute(1, 2, 0)
plt.title("First Testing Sample")
plt.imshow(img)
plt.show()